In [1]:
import pandas as pd

# Path to the Parquet file
file_path = 'Tavg_common_points.parquet'

# Load the Parquet file
data = pd.read_parquet(file_path)

# Get column names
column_names = data.columns

# Print column names
print("Column Names:")
for col in column_names:
    print(col)


Column Names:
x
y
1991_Autumn
1991_Winter
1991_Spring
1991_Summer
1992_Autumn
1992_Winter
1992_Spring
1992_Summer
1993_Autumn
1993_Winter
1993_Spring
1993_Summer
1994_Autumn
1994_Winter
1994_Spring
1994_Summer
1995_Autumn
1995_Winter
1995_Spring
1995_Summer
1996_Autumn
1996_Winter
1996_Spring
1996_Summer
1997_Autumn
1997_Winter
1997_Spring
1997_Summer
2009_Autumn
2009_Winter
2009_Spring
2009_Summer
2010_Autumn
2010_Winter
2010_Spring
2010_Summer
2011_Autumn
2011_Winter
2011_Spring
2011_Summer
2012_Autumn
2012_Winter
2012_Spring
2012_Summer
2013_Autumn
2013_Winter
2013_Spring
2013_Summer
2014_Autumn
2014_Winter
2014_Spring
2014_Summer
2015_Autumn
2015_Winter
2015_Spring
2015_Summer
2016_Autumn
2016_Winter
2016_Spring
2016_Summer
2017_Autumn
2017_Winter
2017_Spring
2017_Summer
2018_Autumn
2018_Winter
2018_Spring
2018_Summer
2019_Autumn
2019_Winter
2019_Spring
2019_Summer


In [10]:
import pandas as pd
import statsmodels.api as sm
import re
import matplotlib.pyplot as plt

def sliding_kernel_ols_v6(
    rain_data_path, 
    z_score_data_path,
    kernel_sizes,          # A list of kernel sizes, e.g. [1, 3, 5]
    start_year,
    end_year,
    season_order=None
):
    """
    Runs sliding kernel OLS for each kernel size in 'kernel_sizes',
    saves CSV results for each, and creates a single line graph
    with R-squared vs. response variable (sorted from the largest year to the smallest).

    Args:
        rain_data_path (str): Path to the rain (or Tavg) parquet file.
        z_score_data_path (str): Path to the z-score parquet file.
        kernel_sizes (list[int]): List of kernel sizes (e.g., [1, 3, 5]).
        start_year (int): Start year for filtering (inclusive).
        end_year (int): End year for filtering (inclusive).
        season_order (list[str], optional): Custom season order if desired. 
                                            Defaults to ['Autumn','Winter','Spring','Summer'].
    """
    
    # Default season order if none provided
    if season_order is None:
        season_order = ['Autumn', 'Winter', 'Spring', 'Summer']

    # ------------------- 1. LOAD THE DATA -------------------
    rain_data = pd.read_parquet(rain_data_path)
    z_score_data = pd.read_parquet(z_score_data_path)
    
    # Align indices
    rain_data = rain_data.set_index(['x', 'y'])
    z_score_data = z_score_data.set_index(['x', 'y'])
    
    # Helper function to parse column names into (year, season).
    # Returns None if it doesn't match the (start_year <= year <= end_year) condition.
    def parse_column_name(col_name):
        match = re.match(r'(\d{4})_(\w+)', col_name)
        if match:
            year = int(match.group(1))
            season = match.group(2)
            if start_year <= year <= end_year:
                return (year, season)
        return None
    
    # ------------------- 2. PREPARE LIST OF VALID COLUMNS -------------------
    # Filter columns that parse successfully and fall within [start_year, end_year].
    valid_columns = [col for col in rain_data.columns if parse_column_name(col)]
    
    # Sort columns by ascending (year, season), then reverse to get descending
    valid_columns = sorted(
        valid_columns, 
        key=lambda c: (
            parse_column_name(c)[0],                      # year
            season_order.index(parse_column_name(c)[1])   # season index
        )
    )
    valid_columns.reverse()  # Now the first in the list is the largest (year, season)

    # This list will store DataFrames for each kernel size so we can plot them together.
    aggregated_results = []

    # ------------------- 3. LOOP OVER KERNEL SIZES -------------------
    for k_size in kernel_sizes:
        print(f"\n--- Running sliding kernel OLS with kernel_size={k_size} ---")
        
        # We'll accumulate rows into csv_rows for the CSV file
        csv_rows = []

        # Iterate over valid_columns as potential "response"
        for response_idx, response_column in enumerate(valid_columns):
            # Parse (year, season) from the column
            response_parsed = parse_column_name(response_column)
            if response_parsed is None:
                continue  # Skip if it doesn't match the parse criteria

            # Collect the predictors based on the kernel size, going *forward* in valid_columns
            # from the current response_idx.
            predictors = []
            for offset in range(k_size):
                # Check if there's a valid column at response_idx + offset
                if (response_idx + offset) < len(valid_columns):
                    predictors.append(valid_columns[response_idx + offset])
                else:
                    break  # Not enough columns left to satisfy the kernel size
            
            # If we don't have enough predictors, break out of the loop.
            if len(predictors) < k_size:
                print(
                    f"Stopping at {response_column}: "
                    f"Not enough predictors for kernel size {k_size}."
                )
                break
            
            # Prepare response (y) and predictor matrix (X)
            y = z_score_data[response_column]
            X = rain_data[predictors].copy()
            X = sm.add_constant(X, has_constant='add')  # Add constant for OLS intercept

            # Fit OLS model
            model = sm.OLS(y, X, missing='drop').fit()
            
            # Store one row per predictor, though the R-squared is the same for them all
            for idx_p, predictor_col in enumerate(predictors):
                csv_rows.append({
                    'response_column': response_column,
                    'kernel_position': idx_p + 1,  # 1-based index
                    'predictor': predictor_col,
                    'r_squared': model.rsquared
                })
        
        # Convert results to a DataFrame and export to CSV
        results_df = pd.DataFrame(csv_rows)
        csv_filename = f"sliding_kernel_ols_results_k{k_size}.csv"
        results_df.to_csv(csv_filename, index=False)
        print(f"Results (kernel_size={k_size}) saved to {csv_filename}")

        # Keep a single R-squared per response_column in a DataFrame for plotting
        # Since the same model produces repeated rows (one per predictor), 
        # we drop duplicates by 'response_column'.
        if not results_df.empty:
            df_plot = results_df.drop_duplicates(subset=['response_column']).copy()
            df_plot['kernel_size'] = k_size
            # We'll store only (response_column, kernel_size, r_squared) for the final plot
            aggregated_results.append(df_plot[['response_column', 'kernel_size', 'r_squared']])

    # ------------------- 4. COMBINED LINE PLOT (DESCENDING YEAR) -------------------
    if aggregated_results:
        # Combine all partial DataFrames
        full_plot_df = pd.concat(aggregated_results, ignore_index=True)

        # We define a custom sorting function that sorts from the largest year to the smallest
        def descending_sort_key(resp):
            """
            Returns a tuple for sorting in descending order by:
            - year (largest first)
            - season index (largest index first, if you want 'Summer' before 'Autumn', etc.)
            """
            parsed = parse_column_name(resp)  # e.g. (2017, 'Autumn')
            if parsed is not None:
                year, season = parsed
                # Negate the year, so bigger years become "smaller" in the sort key
                # Negate the season index to invert that as well
                return (-year, -season_order.index(season))
            # If somehow it doesn't parse, push it to the end:
            return (999999, 999999)

        # Add a column with the sort key
        full_plot_df['sort_tuple'] = full_plot_df['response_column'].apply(descending_sort_key)
        # Sort in ascending order by this negative tuple => effectively largest year first
        full_plot_df.sort_values(by='sort_tuple', inplace=True)

        # Extract the unique response variables in the sorted order
        unique_responses = full_plot_df['response_column'].unique().tolist()

        # Prepare the plot
        plt.figure(figsize=(10, 6))

        # Plot one line for each kernel_size on the same axes
        for k_size in sorted(full_plot_df['kernel_size'].unique()):
            # Subset for the current kernel size
            subset = full_plot_df[full_plot_df['kernel_size'] == k_size].copy()
            # Each row in subset corresponds to a unique response_column, 
            # which we have sorted above.
            x_positions = range(len(subset))  # 0..(N-1)

            # Plot as a line (marker='o' is optional)
            plt.plot(
                x_positions,
                subset['r_squared'],
                marker='o',
                label=f'Kernel = {k_size}'
            )

        # Replace x-axis ticks with the (descending) response_column labels
        plt.xticks(ticks=range(len(unique_responses)), labels=unique_responses, rotation=90)

        plt.xlabel('Response Variable (Newest to Oldest)')
        plt.ylabel('R-squared')
        plt.title('R-squared vs. Response Variable (All Kernel Sizes - Descending Years)')
        plt.legend()
        plt.tight_layout()

        # Save the combined line plot
        combined_plot_filename = "combined_line_plot_descending.png"
        plt.savefig(combined_plot_filename, dpi=150)
        plt.close()
        print(f"\nCombined line plot saved to {combined_plot_filename}")
    else:
        print("\nNo data to plot. Possibly no valid columns or no successful OLS fits.")


# ------------------- EXAMPLE USAGE -------------------
if __name__ == "__main__":
    # Example: you can prompt the user, or just define them directly:
    # user_input = input("Enter kernel sizes (comma-separated), e.g. '1,3,5': ")
    # kernel_sizes = [int(x.strip()) for x in user_input.split(',') if x.strip().isdigit()]

    kernel_sizes = [1, 3, 5,8,7]

    # Define your file paths
    rain_data_path = "Tavg_common_points.parquet"
    z_score_data_path = "z_score_common_points.parquet"

    # Define year filtering
    start_year = 2009
    end_year = 2017
    
    # (Optional) custom season order if you prefer a different sequence
    # season_order = ['Summer', 'Autumn', 'Winter', 'Spring']

    # Run the function
    sliding_kernel_ols_v6(
        rain_data_path=rain_data_path,
        z_score_data_path=z_score_data_path,
        kernel_sizes=kernel_sizes,
        start_year=start_year,
        end_year=end_year
        # season_order=season_order  # uncomment to override default
    )



--- Running sliding kernel OLS with kernel_size=1 ---
Results (kernel_size=1) saved to sliding_kernel_ols_results_k1.csv

--- Running sliding kernel OLS with kernel_size=3 ---
Stopping at 2009_Winter: Not enough predictors for kernel size 3.
Results (kernel_size=3) saved to sliding_kernel_ols_results_k3.csv

--- Running sliding kernel OLS with kernel_size=5 ---
Stopping at 2009_Summer: Not enough predictors for kernel size 5.
Results (kernel_size=5) saved to sliding_kernel_ols_results_k5.csv

--- Running sliding kernel OLS with kernel_size=8 ---
Stopping at 2010_Spring: Not enough predictors for kernel size 8.
Results (kernel_size=8) saved to sliding_kernel_ols_results_k8.csv

--- Running sliding kernel OLS with kernel_size=7 ---
Stopping at 2010_Winter: Not enough predictors for kernel size 7.
Results (kernel_size=7) saved to sliding_kernel_ols_results_k7.csv

Combined line plot saved to combined_line_plot_descending.png
